# 04 · Finalise — figures, significance, daily 2023 pipeline and dashboard data, from the trained models
Free Colab is enough: nothing is trained here. **Runtime → Change runtime type → T4 GPU** is faster (the published
method's 3D network is slow on CPU); a CPU runtime works too. Reads `MyDrive/OceanEmbed/pack.tar` (data) and
`MyDrive/OceanEmbed/runpod_results/` (trained models, ensemble reconstruction); writes back into `runpod_results/`.
Every step logs to Drive and can be re-run on its own.

In [ ]:
# ── Setup: Drive (data) + GitHub (code) + deps ──────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/OceanEmbed'          # data, checkpoints, outputs live here
REPO = '/content/Sylithe_Ocean_Embedding'
import os, sys; os.makedirs(ROOT, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/karan27-dev/Sylithe_Ocean_Embedding {REPO}
sys.path.insert(0, REPO)
for m in [m for m in sys.modules if m.startswith('oceanembed')]:
    del sys.modules[m]                              # re-running this cell picks up freshly pulled code
!git -C {REPO} log --oneline -1
# Install only what this Colab image lacks (images differ between sessions), one package at a time, so pip
# never searches a large version space (that is what hung before).
import importlib, importlib.util, subprocess
NEED = {'xarray': 'xarray', 'zarr': 'zarr', 'dask': 'dask', 'netCDF4': 'netCDF4', 'scipy': 'scipy',
        'ee': 'earthengine-api', 'copernicusmarine': 'copernicusmarine', 'earthaccess': 'earthaccess', 'psutil': 'psutil'}
for mod, pkg in NEED.items():
    if importlib.util.find_spec(mod) is None:
        print('installing', pkg)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', pkg], check=True)
importlib.invalidate_caches()
for mod in NEED:
    print(f'{mod:17s}', getattr(importlib.import_module(mod), '__version__', 'ok'))

In [ ]:
# ── Data and trained models onto the local disk (~10 min) ────────────────────
import shutil, glob
RES, DATA, WORK = f'{ROOT}/runpod_results', '/content/data', '/content/root'
os.makedirs(DATA, exist_ok=True); os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f'{DATA}/target.zarr'):
    print('copying pack.tar'); shutil.copy(f'{ROOT}/pack.tar', '/content/pack.tar')
    !cd {REPO} && python -m oceanembed.pack unpack --tar /content/pack.tar --dst {DATA}
    os.remove('/content/pack.tar')
if not os.path.exists(f'{DATA}/hycom.zarr'):
    print('copying hycom.zarr'); shutil.copytree(f'{ROOT}/hycom.zarr', f'{DATA}/hycom.zarr')
shutil.copy(f'{ROOT}/argo_2023.parquet', f'{DATA}/argo_2023.parquet')
for f in ['stats_v2.npz', 'OceanEmbed_NIO_T_2023.nc']:
    if not os.path.exists(f'{WORK}/{f}'):
        print('copying', f); shutil.copy(f'{RES}/{f}', f'{WORK}/{f}')
for d in ['checkpoints', 'baselines']:
    shutil.copytree(f'{RES}/{d}', f'{WORK}/{d}', dirs_exist_ok=True)
ENS = f'{WORK}/OceanEmbed_NIO_T_2023.nc'
print('seeds:', sorted(glob.glob(f'{WORK}/checkpoints/oceanembed_w15/seed*/glorys_best.pt')))

In [ ]:
# ── Leaderboard labels: same numbers, the model's name ───────────────────────
for f in glob.glob(f'{RES}/leaderboard/leaderboard*'):
    t = open(f, encoding='utf-8').read()
    open(f, 'w', encoding='utf-8').write(t.replace('OceanEmbed (', 'Sylithe Ocean Model ('))
print(open(f'{RES}/leaderboard/leaderboard.md', encoding='utf-8').read())

## Daily pipeline: every day of 2023 predicted once and cached, plus the Daily-page files (~10 min)

In [ ]:
!cd {REPO} && python -m oceanembed.daily run --data {DATA} --root {WORK} --start 2023-01-01 --end 2023-12-31 --from-nc {ENS} 2>&1 | tee {RES}/daily_colab.log
!cd {REPO} && python -m oceanembed.daily export --data {DATA} --root {WORK} --out {WORK}/web_ops --maps-last 31 2>&1 | tee -a {RES}/daily_colab.log
for d in ['ops', 'web_ops']:
    shutil.copytree(f'{WORK}/{d}', f'{RES}/{d}', dirs_exist_ok=True)

## Dashboard export (maps for May 2023, Argo scores, embedding regimes)

In [ ]:
!cd {REPO} && python -m oceanembed.run web --data {DATA} --root {WORK} --workers 2 2>&1 | tee {RES}/web_colab.log
shutil.copytree(f'{WORK}/web_export', f'{RES}/web_export', dirs_exist_ok=True)

## Paper evaluation: every metric, the significance test and all 20 figures (longest step)

In [ ]:
!cd {REPO} && python -m oceanembed.paper_eval --data {DATA} --root {WORK} --out {WORK}/paper --ensemble-nc {ENS} --figures 2>&1 | tee {RES}/paper_colab.log
shutil.copytree(f'{WORK}/paper', f'{RES}/paper', dirs_exist_ok=True, ignore=shutil.ignore_patterns('_work'))
print(open(f'{WORK}/paper/significance.csv').read())